# Reconhecimento de Locutor (Speaker Recognition) — Comparação de Métodos

Objetivo: **aplicação embarcada** com speakers cadastrados, reconhecimento de quem está falando
e possibilidade de **cadastrar (enrolar) um novo speaker** armazenando o modelo compacto extraído
da voz (coeficientes MFCC).

O notebook compara **7 métodos** de diferenciação de locutores na mesma base, avalia identificação
(quem fala?) e verificação (está falando a pessoa X?) com limiar, simula uma matrícula de novo
speaker e dimensiona o custo de armazenamento no microcontrolador.

> ⚠️ **Amostra pequena**: só temos 2 falantes (A e B). Os resultados servem para escolher
> arquitetura e *framework* de avaliação — a validação final exige gravações reais de muitos locutores.


## 1. Conceitos

- **Identificação (closed-set)**: dado um áudio, dizer QUEM fala (1 de N cadastrados).
- **Verificação (verification)**: dado um áudio + identidade alegada, aceitar/rejeitar (open-set).
  Aqui entra o **limiar** de decisão.
- **Matrícula (enrollment)**: cadastrar um NOVO speaker — extraímos os MFCC e treinamos/adaptamos
  um modelo compacto (ex.: GMM de poucas gaussianas) que é armazenado a bordo.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import librosa
import scipy.signal as signal

from sklearn.mixture import GaussianMixture
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve
from scipy.spatial.distance import cdist

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (14, 4), 'axes.titlesize': 12,
                     'axes.labelsize': 10, 'font.size': 10})

AUDIO_DIR = Path('audio_exemplo')

# Parâmetros herdados da EDA
TARGET_SR = 16000
VOICE_LOW, VOICE_HIGH = 80, 8000
N_FFT, HOP_LENGTH = 1024, 256
N_MFCC = 13            # 13 MFCC + 13 Δ + 13 ΔΔ = 39 dimensões/frame
SEG_DUR = 2.0          # duração dos segmentos de análise
SEEDS = 15             # repetições Monte Carlo da avaliação

# Descoberta automática da base em audio_exemplo/.
# Convenção nova:  locutor_<nome>_<sessao>.wav  -> sessões do mesmo locutor são agrupadas
# Compatível com a base antiga:                            audio_<nome>.wav/.ogg
import re
LOCUTOR_RE = re.compile(r'^locutor_(?P<nome>.+?)_(?P<sessao>s\d+)$')
LEGACY_RE  = re.compile(r'^audio_(?P<nome>.+)$')

SPEAKER_FILES = {}
for f in sorted(AUDIO_DIR.iterdir()):
    if f.suffix.lower() not in ('.wav', '.ogg', '.mp3', '.flac'):
        continue
    m = LOCUTOR_RE.match(f.stem)
    if m:
        SPEAKER_FILES.setdefault(m.group('nome'), []).append(f)
        continue
    m = LEGACY_RE.match(f.stem)
    if m:
        SPEAKER_FILES.setdefault(m.group('nome'), []).append(f)
        continue
    SPEAKER_FILES.setdefault(f.stem, []).append(f)

for sp in SPEAKER_FILES:
    SPEAKER_FILES[sp] = sorted(SPEAKER_FILES[sp])

if not SPEAKER_FILES:
    raise SystemExit(f'Nenhum áudio em {AUDIO_DIR}/')

print('Locutores descobertos:', {sp: len(v) for sp, v in SPEAKER_FILES.items()})
print('Setup OK — parâmetros:', dict(TARGET_SR=TARGET_SR, N_MFCC=39, SEG_DUR=SEG_DUR))


## 2. Pré-processamento e Extração de MFCC

Mesma cadeia da EDA: **subamostragem 16 kHz → filtro passa-banda 80–8000 Hz → MFCC 39-dim**.
Arquivos do mesmo locutor (`locutor_<nome>_s*.wav`) são concatenados para dar mais dados.


In [ ]:
def preprocess(path):
    y, sr = librosa.load(path, sr=None, mono=True)
    if sr != TARGET_SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)
    nyq = TARGET_SR / 2.0
    b, a = signal.butter(5, [VOICE_LOW / nyq, min(VOICE_HIGH / nyq, 0.99)], btype='bandpass')
    y = signal.filtfilt(b, a, y)
    return y.astype(np.float32)

def extract_mfccs(y):
    mfcc = librosa.feature.mfcc(y=y, sr=TARGET_SR, n_mfcc=N_MFCC,
                                n_fft=N_FFT, hop_length=HOP_LENGTH)
    d1 = librosa.feature.delta(mfcc)
    d2 = librosa.feature.delta(mfcc, order=2)
    return np.vstack([mfcc, d1, d2])   # (39, T)

signals = {}
mfccs = {}
for sp, files in SPEAKER_FILES.items():
    y = np.concatenate([preprocess(p) for p in files])
    signals[sp] = y
    mfccs[sp] = extract_mfccs(y)
    print(f'{sp:7s}  duração total={len(y)/TARGET_SR:6.2f}s  frames MFCC={mfccs[sp].shape[1]}  dim={mfccs[sp].shape[0]}')


## 3. Segmentação temporal

Audio contínuo é picado em **segmentos de 2s**. Cada segmento é uma "amostra" independente no nosso
conjunto de dados (mimicando o uso real: o sistema analisa janelas de voz e decide quem está falando).


In [ ]:
def make_segments(mfcc, seg_dur=SEG_DUR):
    frames_per_seg = int(round(seg_dur * TARGET_SR / HOP_LENGTH))
    n = mfcc.shape[1]
    return [mfcc[:, i:i + frames_per_seg]
            for i in range(0, n - frames_per_seg + 1, frames_per_seg)]

segments, speaker_ids = [], []
for sp, m in mfccs.items():
    segs = make_segments(m)
    segments.extend(segs)
    speaker_ids.extend([sp] * len(segs))
    print(f'{sp:7s} -> {len(segs)} segmentos de {SEG_DUR}s')

speaker_ids = np.array(speaker_ids)
SPEAKERS = sorted(np.unique(speaker_ids))
print(f'Total: {len(segments)} segmentos | locutores: {SPEAKERS}')

# Normalização do cepstrum (CMN): remove desvio de canal/microfone por segmento.
# Aplicada só nos métodos baseados em frames (GMM/GMM-UBM/DTW).
def cmn(seg):
    return seg - seg.mean(axis=1, keepdims=True)

segments_cmn = [cmn(s) for s in segments]

# Representação vetorial por segmento (média temporal dos 39 MFCC) p/ métodos vetoriais.
mean_vecs = np.array([s.mean(axis=1) for s in segments])
print(f'mean_vecs shape: {mean_vecs.shape}')


## 4. Divisão treino/teste (estratificada)

Como os segmentos de cada locutor são contínuos (frames correlacionados), avaliamos de forma robusta
com **split aleatório repetido (Monte Carlo, `SEEDS` iterações)** e reportamos média±desvio da acurácia.


In [ ]:
def make_split(seed):
    tr, te = train_test_split(np.arange(len(segments)), test_size=0.3,
                              stratify=speaker_ids, random_state=seed)
    return tr, te

# Exemplo visual do split (seed 0)
tr, te = make_split(0)
print(f'seed=0 -> treino {len(tr)}, teste {len(te)}')
for sp in SPEAKERS:
    n_tr = (speaker_ids[tr] == sp).sum()
    n_te = (speaker_ids[te] == sp).sum()
    print(f'  {sp}: treino={n_tr}  teste={n_te}')


## 5. Os 7 métodos comparados

| # | Método | Tipo | Ideia |
|---|--------|------|-------|
| 1 | **Centroide + cosseno** | vetorial | Média temporal dos MFCC de cada locutor; classifica pela maior similaridade de cosseno. *Zero treinamento.* |
| 2 | **k-NN** | vetorial | Vizinhança k=3 dos segmentos de treino no espaço de MFCC (cosseno). |
| 3 | **GMM por locutor** | frames | 1 GMM diagonal (4 gaussianas) por locutor, treinado nos frames MFCC; score = log-verossimilhança. |
| 4 | **GMM-UBM (MAP)** | frames | UBM global + **adaptação MAP** por locutor; score = **LLR contra o UBM** (remove canal/sessão). Ideal p/ embarcado e p/ matrícula. |
| 5 | **SVM (RBF)** | vetorial | Hiperplano de fronteira entre locutores no vetor-médio MFCC. |
| 6 | **MLP** | vetorial | Rede neural pequena (1 camada oculta 64) no vetor-médio MFCC. |
| 7 | **DTW** | trajetórias | Alinha temporalmente as trajetórias MFCC de 2 segmentos; 1-NN pela menor distorção. |


In [ ]:
# ---------- Utilidades: MAP adaptation e construção de GMM a partir de parâmetros ----------
def gmm_from_params(weights, means, covs):
    g = GaussianMixture(n_components=len(weights), covariance_type='diag')
    g.weights_, g.means_, g.covariances_ = weights, means, covs
    g.precisions_cholesky_ = 1.0 / np.sqrt(covs)
    g.n_features_in_ = means.shape[1]
    return g

def map_adapt(ubm, X, tau=16.0):
    resp = ubm.predict_proba(X)
    nk = resp.sum(0)
    mean_k = (resp.T @ X) / nk[:, None]
    sq_k = resp.T @ (X ** 2)
    var_k = sq_k / nk[:, None] - mean_k ** 2
    alpha = nk / (nk + tau)
    means = alpha[:, None] * mean_k + (1 - alpha[:, None]) * ubm.means_
    covs = (alpha[:, None] * (var_k + mean_k ** 2)
            + (1 - alpha[:, None]) * (ubm.covariances_ + ubm.means_ ** 2) - means ** 2)
    covs = np.clip(covs, 1e-6, None)
    weights = alpha * (nk / nk.sum()) + (1 - alpha) * ubm.weights_
    weights = weights / weights.sum()
    return gmm_from_params(weights, means, covs)

def frames_of(X, cls, y):
    return np.concatenate([seg for seg, lab in zip(X, y) if lab == cls], axis=1).T


In [ ]:
# ---------- Implementação dos 7 modelos ----------
class CentroidModel:
    def fit(self, X, y):
        self.classes_ = np.unique(np.asarray(y))
        self.W_ = np.array([np.asarray(X)[np.asarray(y) == c].mean(0) for c in self.classes_])
        return self
    def predict(self, X):
        X = np.asarray(X); X = X / np.linalg.norm(X, axis=1)[:, None]
        W = self.W_ / np.linalg.norm(self.W_, axis=1)[:, None]
        return np.array([self.classes_[i] for i in np.argmax(X @ W.T, axis=1)])

class GMMModel:
    def __init__(self, n_components=4): self.n_components = n_components
    def fit(self, X, y):
        self.models_ = {}
        for c in np.unique(np.asarray(y)):
            self.models_[c] = GaussianMixture(self.n_components, covariance_type='diag',
                                              random_state=0).fit(frames_of(X, c, np.asarray(y)))
        return self
    def score_map(self, seg):
        return {c: m.score_samples(seg.T).mean() for c, m in self.models_.items()}
    def predict(self, X):
        return np.array([max(self.score_map(s).items(), key=lambda kv: kv[1])[0] for s in X])

class GMMUBMModel(GMMModel):
    # GMM-UBM com adaptacao MAP. O score e o log-likelihood ratio (LLR) contra o UBM,
    # cancelando o ruido comum de canal/sessao — a verificacao fica estavel.
    def __init__(self, n_components=4, tau=16.0):
        super().__init__(n_components); self.tau = tau
    def fit(self, X, y):
        y = np.asarray(y)
        all_frames = np.concatenate(X, axis=1).T
        self.ubm_ = GaussianMixture(self.n_components, covariance_type='diag',
                                    random_state=0).fit(all_frames)
        self.models_ = {c: map_adapt(self.ubm_, frames_of(X, c, y), self.tau)
                        for c in np.unique(y)}
        return self
    def score_map(self, seg):
        ll_ubm = self.ubm_.score_samples(seg.T).mean()
        return {c: m.score_samples(seg.T).mean() - ll_ubm for c, m in self.models_.items()}

class DTWModel:
    def fit(self, X, y):
        self.X_ = list(X); self.y_ = np.asarray(y)
        return self
    def predict(self, X):
        preds = []
        for seg in X:
            ds = [dtw_distance(seg, ref) for ref in self.X_]
            preds.append(self.y_[int(np.argmin(ds))])
        return np.array(preds)

def dtw_distance(a, b):
    D = cdist(a.T, b.T, metric='sqeuclidean')
    T1, T2 = D.shape
    C = np.zeros_like(D)
    C[0, 0] = D[0, 0]
    for i in range(1, T1): C[i, 0] = C[i-1, 0] + D[i, 0]
    for j in range(1, T2): C[0, j] = C[0, j-1] + D[0, j]
    for i in range(1, T1):
        for j in range(1, T2):
            C[i, j] = D[i, j] + min(C[i-1, j], C[i, j-1], C[i-1, j-1])
    return C[-1, -1]

METHODS = [
    ('1. Centroide + cosseno', CentroidModel(), 'mean'),
    ('2. k-NN (k=3, cosseno)', KNeighborsClassifier(n_neighbors=3, weights='distance', metric='cosine'), 'mean'),
    ('3. GMM por locutor (4 gauss)', GMMModel(n_components=4), 'frame'),
    ('4. GMM-UBM (MAP, tau=16)', GMMUBMModel(n_components=4, tau=16.0), 'frame'),
    ('5. SVM (RBF)', SVC(C=1.0, kernel='rbf', gamma='scale'), 'mean'),
    ('6. MLP (64 neurônios)', MLPClassifier(hidden_layer_sizes=(64,), max_iter=3000, random_state=0), 'mean'),
    ('7. DTW (1-NN)', DTWModel(), 'frame'),
]
DATA = {'mean': (mean_vecs, speaker_ids), 'frame': (segments_cmn, speaker_ids)}
print(f'{len(METHODS)} métodos registrados.')


## 6. Avaliação — Identificação de Locutor (Monte Carlo)

Para cada um dos `SEEDS` splits: treina o modelo, classifica os segmentos de teste, mede acurácia.


In [ ]:
results = defaultdict(list)
for seed in range(SEEDS):
    tr, te = make_split(seed)
    for name, model, kind in METHODS:
        X, y = DATA[kind]
        X_tr = X[tr] if kind == 'mean' else [X[i] for i in tr]
        X_te = X[te] if kind == 'mean' else [X[i] for i in te]
        model.fit(X_tr, y[tr])
        pred = model.predict(X_te)
        results[name].append(accuracy_score(y[te], pred))

print(f'{"Método":28s} {"Acurácia média":>16s} {"±":>2s} {"Desvio":>6s}')
print('-' * 58)
summary = []
for name, _, _ in METHODS:
    acc = np.mean(results[name]); std = np.std(results[name])
    summary.append((name, acc, std))
    print(f'{name:28s} {acc*100:14.1f}% ± {std*100:5.1f}%')


In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.5))
names = [s[0] for s in summary]
means = [s[1]*100 for s in summary]
stds  = [s[2]*100 for s in summary]
colors = plt.cm.viridis(np.linspace(0.25, 0.95, len(names)))
bars = ax.barh(names[::-1], means[::-1], xerr=stds[::-1], color=colors[::-1],
               capsize=4, edgecolor='black', linewidth=0.5)
best = np.argmax(means)
bars[len(names)-1-best].set_edgecolor('crimson'); bars[len(names)-1-best].set_linewidth(2.5)
ax.set_xlabel('Acurácia de identificação (%)')
ax.set_title(f'Comparação de métodos — média ± desvio sobre {SEEDS} splits (closed-set)', fontsize=13)
for i, (m, s) in enumerate(zip(means[::-1], stds[::-1])):
    ax.text(m + 1, i, f'{m:.1f}% ± {s:.1f}', va='center', fontsize=9)
ax.set_xlim(0, 110)
plt.tight_layout(); plt.show()


## 7. Matriz de confusão (melhor método)

Rodamos o melhor método em média com um split fixo (seed 0) e detalhamos quem confundiu com quem.


In [ ]:
best_name = summary[np.argmax([s[1] for s in summary])][0]
best_model = [m for n, m, _ in METHODS if n == best_name][0]
kind = [k for n, _, k in METHODS if n == best_name][0]

tr, te = make_split(0)
X, y = DATA[kind]
X_tr = X[tr] if kind == 'mean' else [X[i] for i in tr]
X_te = X[te] if kind == 'mean' else [X[i] for i in te]
best_model.fit(X_tr, y[tr])
pred = best_model.predict(X_te)

cm = confusion_matrix(y[te], pred, labels=SPEAKERS)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(SPEAKERS))); ax.set_xticklabels(SPEAKERS)
ax.set_yticks(range(len(SPEAKERS))); ax.set_yticklabels(SPEAKERS)
ax.set_xlabel('Predito'); ax.set_ylabel('Real')
ax.set_title(f'Matriz de confusão — {best_name}')
for i in range(len(SPEAKERS)):
    for j in range(len(SPEAKERS)):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black')
fig.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()
print(f'{best_name}: acurácia = {accuracy_score(y[te], pred)*100:.1f}%')


## 8. Verificação com Limiar (EER)

Identificação é closed-set; na prática o sistema precisa **aceitar ou rejeitar** (ex.: "é o locutor A
mesmo falando?"). Com o **GMM-UBM**, usamos o **LLR (log-likelihood ratio) contra o UBM** como score:

- **score genuíno**: LLR quando o segmento vem da pessoa correta;
- **score impostor**: LLR médio contra os modelos das OUTRAS pessoas.

Valores **positivos → voz combina com o modelo**; negativos → não combina. Agregamos os scores de
todos os `SEEDS` splits (estatística mais estável que 1 split só) e traçamos a ROC com o **EER**
(limiar em que falso-aceite = falso-rejeite).


In [ ]:
genuine, impostor = [], []
for seed in range(SEEDS):
    tr, te = make_split(seed)
    m = GMMUBMModel(n_components=4, tau=16.0)
    m.fit([segments_cmn[i] for i in tr], speaker_ids[tr])
    for seg, true_sp in zip([segments_cmn[i] for i in te], speaker_ids[te]):
        sc = m.score_map(seg)                      # LLR vs UBM
        genuine.append(sc[true_sp])
        impostor.extend([v for k, v in sc.items() if k != true_sp])

y_true = np.array([1]*len(genuine) + [0]*len(impostor))
scores = np.array(genuine + list(impostor))
fpr, tpr, thr = roc_curve(y_true, scores)
fnr = 1 - tpr
eer_idx = int(np.argmin(np.abs(fpr - fnr)))
EER = (fpr[eer_idx] + fnr[eer_idx]) / 2
THRESHOLD = float(thr[eer_idx])

print(f'Genuínos: {len(genuine)}  Impostores: {len(impostor)}  (agregados de {SEEDS} splits)')
print(f'EER = {EER*100:.1f}%  |  Limiar LLR (EER) = {THRESHOLD:.2f}')
print(f'Score genuíno médio = {np.mean(genuine):+.2f} | impostor médio = {np.mean(impostor):+.2f}')


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.5))

ax1.hist(genuine, bins=25, alpha=0.7, color='steelblue', label='Genuíno (mesma pessoa)')
ax1.hist(impostor, bins=25, alpha=0.7, color='crimson', label='Impostor (outra pessoa)')
ax1.axvline(THRESHOLD, color='black', linestyle='--', linewidth=2, label=f'Limiar (EER) = {THRESHOLD:.2f}')
ax1.set_xlabel('Score (LLR vs UBM)'); ax1.set_ylabel('Frequência')
ax1.set_title('Distribuição dos scores (LLR) — GMM-UBM'); ax1.legend()

ax2.plot(fpr, tpr, color='darkorange', linewidth=2)
ax2.plot([0, 1], [0, 1], '--', color='gray')
ax2.scatter(fpr[eer_idx], tpr[eer_idx], color='crimson', s=60, zorder=5,
            label=f'EER = {EER*100:.1f}%')
ax2.set_xlabel('Falso positivo (falso-aceite)') ; ax2.set_ylabel('Verdadeiro positivo')
ax2.set_title('Curva ROC — verificação de locutor'); ax2.legend()
plt.tight_layout(); plt.show()


## 9. Pipeline de Matrícula (enroll) de um Novo Speaker

Num embarcado, ao cadastrar um novo usuário:

1. grava-se um trecho de voz (ex.: 10–20s supervisionado);
2. extrai-se MFCC (39-dim);
3. **adapta-se o UBM global via MAP** → modelo individual (centenas de floats!);
4. salva-se o modelo + limiar no flash.

Abaixo simulamos: o segundo locutor era desconhecido, é matriculado agora e depois verificado
(aceitação do próprio áudio e rejeição de áudio de outro). Usamos o UBM treinado só com o primeiro locutor
(como se fosse o "fundo" do sistema).


In [ ]:
# ---- Etapa 1: UBM treinado com o locutor JÁ cadastrado (não inclui o novo) ----
BASE = SPEAKERS[0]
NEW  = SPEAKERS[1] if len(SPEAKERS) >= 2 else SPEAKERS[0]
base_idx = np.where(speaker_ids == BASE)[0]
ubm_sys = GaussianMixture(n_components=4, covariance_type='diag', random_state=0).fit(
    np.concatenate([segments_cmn[i] for i in base_idx], axis=1).T)
print('UBM do sistema treinado com o locutor já cadastrado:', BASE)

# ---- Etapa 2: matricular os novos locutores (MAP adaptation sobre o UBM) ----
def enroll(name, path_list, ubm, tau=16.0):
    y = np.concatenate([preprocess(p) for p in path_list])
    mfcc = cmn(extract_mfccs(y))
    model = map_adapt(ubm, mfcc.T, tau)
    n_bytes = (model.weights_.size + model.means_.size + model.covariances_.size) * model.means_.dtype.itemsize
    return name, model, n_bytes

name, new_model, nbytes = enroll(NEW, SPEAKER_FILES[NEW], ubm_sys)
print(f'Matrícula concluída: {name} | modelo GMM armazenado = {nbytes} bytes')
print(f'  ({new_model.means_.shape[0]} gaussianas × {new_model.means_.shape[1]} dims, covariância diagonal)')


In [ ]:
# ---- Etapa 3: verificar áudios contra o modelo recém-cadastrado ----
# Score = LLR contra o UBM do sistema: positivo = voz combina, negativo = não combina.
def verify(path, model, ubm):
    X = cmn(extract_mfccs(preprocess(path))).T
    return float(model.score_samples(X).mean() - ubm.score_samples(X).mean())

new_file   = SPEAKER_FILES[NEW][0]    # áudio do locutor recém-matriculado
base_file  = SPEAKER_FILES[BASE][0]   # áudio de quem já estava (impostor)
new_score  = verify(new_file, new_model, ubm_sys)
base_score = verify(base_file, new_model, ubm_sys)

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar([f'{NEW} (genuíno)', f'{BASE} (impostor)'], [new_score, base_score],
              color=['steelblue', 'crimson'], width=0.5)
ax.axhline(THRESHOLD, color='black', linestyle='--', linewidth=2, label=f'Limiar LLR (EER) = {THRESHOLD:.2f}')
ax.axhline(0, color='gray', linestyle=':', linewidth=1.5, label='LLR = 0')
for b, s in zip(bars, [new_score, base_score]):
    ax.text(b.get_x() + b.get_width()/2, s, f'{s:+.2f}', ha='center', va='bottom', fontsize=11)
ax.set_ylabel('Score (LLR vs UBM)'); ax.set_title('Verificação do novo locutor matriculado')
ax.set_ylim(min(base_score, THRESHOLD) - 1.5, max(new_score, THRESHOLD) + 1.5)
ax.legend()
plt.tight_layout(); plt.show()

for label, s in [(NEW, new_score), (BASE, base_score)]:
    decision = 'ACEITO ✔' if s >= THRESHOLD else 'REJEITADO ✘'
    print(f'  {label:7s} LLR={s:+8.2f}  -> {decision}')


## 10. Dimensionamento para o Embarcado

A grande pergunta do MCU: **quanto ocupa cada locutor cadastrado?** Comparamos 3 estratégias:
arquivo de áudio bruto, MFCC crú e o modelo GMM compacto.


In [ ]:
def gmm_bytes(model):
    return (model.weights_.size + model.means_.size + model.covariances_.size) * model.means_.dtype.itemsize

N_SPK = 10  # cenário: 10 locutores cadastrados
raw_bytes = int(np.sum([signals[sp].nbytes for sp in signals]))           # áudios originais (na real: 1 arquivo/locutor)
mfcc_bytes = int(np.sum([mfccs[sp].nbytes for sp in signals]))            # MFCCs de todos os áudios
gmm_bytes_total = N_SPK * gmm_bytes(new_model)                            # 10 GMMs como o do exemplo

print(f'{"Estratégia":35s} {"Bytes/locutor":>18s} {"10 locutores":>16s}')
print('-' * 72)
for lbl, b, n in [('Áudio bruto (48 kHz float32)', raw_bytes/len(signals), raw_bytes),
                  ('MFCC cru (13+Δ+ΔΔ)', mfcc_bytes/len(signals), mfcc_bytes),
                  ('GMM diagonal (4 gauss, 39-dim)', gmm_bytes(new_model), gmm_bytes_total)]:
    print(f'{lbl:35s} {b:16.0f} B {n/(1<<10):14.1f} kB')

fig, ax = plt.subplots(figsize=(9, 4))
labels = ['Áudio bruto', 'MFCC cru', f'GMM compacto ({gmm_bytes(new_model)} B)']
vals = [raw_bytes, mfcc_bytes, gmm_bytes_total]
ax.bar(labels, [v/(1<<10) for v in vals], color=['gray', 'darkorange', 'forestgreen'], width=0.5)
for i, v in enumerate([v/(1<<10) for v in vals]):
    ax.text(i, v + .5, f'{v:.1f} kB', ha='center', fontsize=11)
ax.set_ylabel('Espaço em flash (kB)'); ax.set_title(f'Estimativa para {N_SPK} locutores cadastrados')
plt.tight_layout(); plt.show()
